# Step 3 - Classification training
Pre-trained GoogLeNet family model (InceptionV3 in Keras) with a new classification layer.

## Import libraries

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers

## Hyperparameters

In [ ]:
SPLIT_DIR = Path("../data/classification/processed")
MODEL_DIR = Path("../models/classification")

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

## Load the images by batches
`image_size` resizes every image to 224x224. Only the train set is shuffled.

In [ ]:
def load_dataset(folder, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        folder,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=42,
    )

In [ ]:
train_ds = load_dataset(SPLIT_DIR / "train", shuffle=True)
val_ds = load_dataset(SPLIT_DIR / "val", shuffle=False)

class_names = train_ds.class_names
print(class_names)

## Preprocessing
Convert to float and normalize the pixels from [0, 255] to [-1, 1], like InceptionV3 expects.

In [ ]:
def preprocess(images, labels):
    images = tf.cast(images, tf.float32)
    images = tf.keras.applications.inception_v3.preprocess_input(images)
    return images, labels

In [ ]:
train_ds = train_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)

In [ ]:
images, labels = next(iter(train_ds))
print("Batch shape:", images.shape)
print("Min:", images.numpy().min(), "Max:", images.numpy().max())

## Build the model
The pre-trained part is frozen. We replace the classification layer with a new one for our 4 classes.

In [ ]:
def build_model(num_classes):
    base_model = tf.keras.applications.InceptionV3(
        weights="imagenet",
        include_top=False,
        input_shape=IMAGE_SIZE + (3,),
    )
    base_model.trainable = False

    model = tf.keras.Sequential([
        tf.keras.Input(shape=IMAGE_SIZE + (3,)),
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ])
    return model

In [ ]:
model = build_model(len(class_names))
model.summary()

## Compile
- optimizer: Adam
- loss: sparse categorical crossentropy (labels are integers 0..3)
- metric: accuracy

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

## Train
Early stopping stops when the validation loss does not improve for 3 epochs and keeps the best weights.

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop],
)

## Training curves (train vs validation)

In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric in zip(axes, ["accuracy", "loss"]):
        ax.plot(history.history[metric], label="train")
        ax.plot(history.history["val_" + metric], label="val")
        ax.set_title(metric)
        ax.set_xlabel("epoch")
        ax.legend()
    plt.show()

In [ ]:
plot_history(history)

## Save the model

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save(MODEL_DIR / "googlenet_classifier.keras")

with open(MODEL_DIR / "class_names.json", "w") as file:
    json.dump(class_names, file)